# FlyVis native input mapping

Use FlyVis's native `Stimulus.input_index` mapping to inspect which network node corresponds to each input hexal / receptor channel.

The important native convention is:

```python
stimulus.input_index[receptor_channel, hexal_id] -> node_id
```

where:

- `hexal_id` is the FlyVis input position, i.e. one ommatidium-like / retinotopic column position.
- `receptor_channel` is the input receptor type index: `0..7` corresponds to `R1..R8`.
- `node_id` is the actual node index inside the full FlyVis network buffer.

So, for example, `input_index[0, 0]` means: FlyVis hexal/ommatidium-like point 0, receptor channel R1.

## Where each value comes from in the original system

- `extent=15` comes from the FlyVis network/connectome config, e.g. `flyvis/config/network/connectome/connectome.yaml`, and is passed into `ConnectomeFromAvgFilters(..., extent=15)`. It defines the radius of the hexagonal input grid.
- `721` is not manually specified here; it follows from `extent=15` via the hex-grid size `1 + 3 * extent * (extent + 1)`.
- `input_cell_types` comes from the connectome JSON field `input_units` in `flyvis/connectome/fib25-fib19_v2.2.json`; for this model it is `R1` through `R8`.
- `stimulus.input_index` is constructed by `Stimulus` in `flyvis/network/stimulus.py` from `connectome.nodes.layer_index` for each input cell type.
- `node_id` is the full-network node index used by FlyVis internally in the stimulus buffer of shape `(batch, frames, n_nodes)`.
- `u, v` come from `connectome.nodes.u` and `connectome.nodes.v`; they are FlyVis's native hex-grid coordinates for each node.
- `hexal_id` is the column index along the second axis of `stimulus.input_index`; it is an input-position index, not a separate real 3D eye-surface ID.


In [ ]:
import pandas as pd

from flyvis.connectome import ConnectomeFromAvgFilters
from flyvis.network.stimulus import Stimulus


try:
    # If a Network object already exists in the notebook, use its native objects.
    # Source: flyvis/network/network.py initializes network.connectome and network.stimulus.
    connectome = network.connectome
    stimulus = network.stimulus
except NameError:
    # Otherwise construct the same native connectome/stimulus objects directly.
    # file: original connectome specification with input_units = R1...R8.
    # extent: original/default FlyVis hex-grid radius; extent=15 -> 721 hexals.
    connectome = ConnectomeFromAvgFilters(file="fib25-fib19_v2.2.json", extent=15)
    stimulus = Stimulus(connectome, init_buffer=False)

def clean_name(value):
    """Convert FlyVis byte/string labels to plain Python strings."""
    if hasattr(value, "item"):
        value = value.item()
    if isinstance(value, bytes):
        return value.decode()
    return str(value)


# Source: connectome.input_cell_types, generated from `input_units` in
# flyvis/connectome/fib25-fib19_v2.2.json.
# Usually: ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8'].
input_cell_types = [clean_name(name) for name in connectome.input_cell_types[:]]

# Source: Stimulus.input_index in flyvis/network/stimulus.py.
# It is built as:
#   np.array([connectome.nodes.layer_index[cell_type] for cell_type in input_cell_types])
# Shape is (n_input_receptor_types, n_hexals), normally (8, 721).
# If you pass stim with shape (batch, frames, 8, 721), then:
#   stim[:, :, receptor_channel, hexal_id]
# is written to network node:
#   input_index[receptor_channel, hexal_id]
input_index = stimulus.input_index

input_cell_types, input_index.shape


## Query one point

Use this cell to answer questions like:

> Which FlyVis network node is ommatidium-like point `hexal_id=0`, receptor `R1`?

In zero-based Python indexing:

- ommatidium 1 / first FlyVis input point is `hexal_id = 0`
- receptor 1 / R1 is `receptor_channel = 0`

In [ ]:
# Change these two values to inspect any FlyVis input point.
# hexal_id comes from the input position axis of stimulus.input_index.
# receptor_name must be one of connectome.input_cell_types, e.g. R1...R8.
hexal_id = 0
receptor_name = "R1"

# Convert receptor name ('R1'...'R8') to the channel index used by the input tensor.
# Source: connectome.input_cell_types.
receptor_channel = input_cell_types.index(receptor_name)

# This is the native FlyVis lookup.
# Source: stimulus.input_index.
#   (receptor_channel, hexal_id) -> full network node_id
node_id = int(input_index[receptor_channel, hexal_id])

pd.DataFrame([{
    "hexal_id": hexal_id,
    "receptor_channel": receptor_channel,
    "receptor_name": receptor_name,
    "node_id": node_id,

    # Source: connectome.nodes table.
    # node_type should match receptor_name here, because this is an input node.
    "node_type": clean_name(connectome.nodes.type[node_id]),

    # Source: connectome.nodes.u / connectome.nodes.v.
    # These are FlyVis's original hex-grid coordinates for this node.
    "u": int(connectome.nodes.u[node_id]),
    "v": int(connectome.nodes.v[node_id]),
}])


## Export all input points

This expands the native FlyVis mapping into a human-readable table with one row per input receptor.

For default `extent=15`, this produces:

```text
721 hexal positions × 8 receptor channels = 5768 rows
```

The resulting CSV only exports FlyVis's native mapping fields. It does **not** add RayTracer-style aliases such as `ommatidium` or `receptor_id`.

Rows are ordered by `hexal_id` first, then `receptor_channel`, so each FlyVis input position's R1-R8 channels stay together.

In [ ]:
rows = []

# Iterate by hexal_id first, then receptor_channel, so the CSV looks like:
#   hexal_id 0: R1, R2, ..., R8
#   hexal_id 1: R1, R2, ..., R8
# FlyVis native axes are still:
#   input_index[receptor_channel, hexal_id] -> node_id
n_hexals = input_index.shape[1]
for hexal_id in range(n_hexals):
    for receptor_channel, receptor_name in enumerate(input_cell_types):
        node_id = input_index[receptor_channel, hexal_id]
        node_id = int(node_id)
        rows.append({
            # FlyVis input position: hexal / retinotopic column index.
            # Source: axis 1 of stimulus.input_index.
            "hexal_id": hexal_id,

            # FlyVis input tensor channel. 0..7 corresponds to R1..R8.
            # Source: axis 0 of stimulus.input_index / connectome.input_cell_types.
            "receptor_channel": receptor_channel,

            # Full-network node index used inside FlyVis's stimulus buffer.
            # Source: stimulus.input_index[receptor_channel, hexal_id].
            "node_id": node_id,

            # Source: connectome.nodes table.
            "node_type": clean_name(connectome.nodes.type[node_id]),

            # Native FlyVis hex-grid coordinates for this network node.
            # Source: connectome.nodes.u / connectome.nodes.v.
            "u": int(connectome.nodes.u[node_id]),
            "v": int(connectome.nodes.v[node_id]),
        })

mapping = pd.DataFrame(rows)
mapping.to_csv("flyvis_native_input_mapping_extent15.csv", index=False)

mapping
